# Exercise 0 - PydanticAI and OpenRouter

2. Sentiment analysis

a) Read in reviews.json and extract all texts and store it in a list

# Läser in filen

In [ ]:
import json
from pathlib import Path

with open(Path("data/reviews.json"), encoding="utf-8") as f:
    data = json.load(f)

texts = [item["text"] for item in data]

print(texts[0])

b) Iterate your list and use an agent to give a sentiment.

# Pydantic Model & Create Agent

In [ ]:
from pydantic import BaseModel
from typing import Literal
from pydantic_ai import Agent

# Defieniera schema (Pydantic v2)

class SentimentOutput(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]

# Skapa agent
sentiment_agent = Agent(
    "openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    system_prompt=(
        "Classify sentiment as one of: positive, negative, neutral."
        "Return ONLY valid structured output."

    )
)

async def predict_sentiment(text: str) -> SentimentOutput:
    result = await sentiment_agent.run(text, output_type=SentimentOutput)
    return result.output 

In [ ]:
# Kör inferens(loop)

predictions = []

for text in texts:
    result = await predict_sentiment(text)
    predictions.append(result.sentiment.lower())

predictions[:5]

c) Now validate how many correct you got and compute the accuracy.



Definition:
acc = correct prediction/total samples
	​


In [ ]:
true_labels = [item["expected_sentiment"].lower() for item in data]

correct = sum(
    1 for true, pred in zip(true_labels, predictions)
    if true == pred
)

accuracy = correct / len(true_labels)

print("Correct predictions:", correct)
print("Accuracy:", accuracy)

d) Also compute recall and precision.

Vi gör det per klass (viktigt!).

In [ ]:
from collections import Counter

labels = ["positive", "negative", "neutral"]

def precision_recall(preds, truths, label):
    tp = sum((p == label and t == label) for p, t in zip(preds, truths))
    fp = sum((p == label and t != label) for p, t in zip(preds, truths))
    fn = sum((p != label and t == label) for p, t in zip(preds, truths))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return precision, recall

In [ ]:
for label in labels:
    p, r = precision_recall(predictions, true_labels, label)
    print(f"{label}")
    print(f"  Precision: {p:.2f}")
    print(f"  Recall:    {r:.2f}")

Viktig insikt
- Precision → hur många av dina predictions var rätt
- Recall → hur många av verkliga du fångade

Om du inte kan detta → du kommer faila i MLOps interviews.

e) Create a confusion matrix.

Detta visar exakt var modellen gör fel.

In [ ]:
labels = ["positive", "negative", "neutral"]

confusion_matrix = {
    true_label: {pred_label: 0 for pred_label in labels}
    for true_label in labels
}

for true, pred in zip(true_labels, predictions):
    confusion_matrix[true][pred] += 1

confusion_matrix